In [ ]:
import base64
import gzip
import json
import pickle
import pyrender
import numpy as np
import random
import secrets
import trimesh
import voxeloo
import zlib

from PIL import Image
from dataclasses import dataclass
from functools import cache
from matplotlib import pyplot as plt
from scipy.ndimage import (
    distance_transform_edt as edt,
    gaussian_filter as blur, 
)
from tqdm.notebook import tqdm
from typing import Any, List, Optional, Dict, Tuple
from scipy import ndimage

### Visualization routines

In [ ]:
def heatmap_to_image(heat):
    return Image.fromarray(np.uint8(heat * 255) , "L")

In [ ]:
def visualize_voxels(voxels, wireframe=False):
    if voxels.dtype == bool:
        voxels = np.array([0, 0xffffffff])[voxels.astype(int)]
    
    mesh = voxeloo.voxels.voxels_to_mesh(voxels)
    
    # Convert the mesh into the trimesh format.
    tm = trimesh.Trimesh(
        vertices=mesh.vertices[:, 0:3],
        faces=mesh.triangles,
    )
    
    # Add the vertex colors.
    tm.visual.face_colors = mesh.vertices[mesh.triangles[:, 0], 3:6]
    
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2, 1.0])
    scene.add(
        pyrender.Mesh.from_trimesh(
            tm,
            smooth=False,
            wireframe=wireframe,
        )
    )
    pyrender.Viewer(
        scene,
        use_raymond_lighting=True,
    )

### Noise routines

In [ ]:
def seed_hash(key):
    return np.uint32(zlib.adler32(str(key).encode("utf-8")))

In [ ]:
def domain(shape, scale):
    coords = np.meshgrid(
        *[np.arange(dim) for dim in shape],
        indexing="ij",
    )
    return tuple(coord / scale for coord in coords)[::-1]

In [ ]:
@cache 
def gen(seed=None):
    return voxeloo.noise.SimplexNoise(seed_hash(seed or 1))
    
def noise(coords, seed=None):
    return gen(seed).noise(coords.reshape(-1, coords.shape[-1])).reshape(*coords.shape[0:-1])

In [ ]:
def explicit_noise(shape, period, weights, lacunarity=2.0, transform=None, seed=None):
    ret = np.zeros(shape, dtype=float)
    for i, w in enumerate(weights):
        coords = np.stack(domain(shape, period / lacunarity**i), axis=-1)
        if transform:
            coords = transform(coords)
        ret += w * noise(coords, seed)
    return ret

In [ ]:
def normalize(nm):
    return (nm - nm.min()) / (nm.max() - nm.min())

### Merge undeground with surface

In [ ]:
@dataclass
class TerrainVoxel:
    name: str
    index: int
    color: int

In [ ]:
def to_flora_id(terrain_id):
    return (1 << 24) | (terrain_id & 0xffffff)

# These IDs match the terrain IDs in: src/galois/js/assets/blocks.ts
terrain = [
    # Blocks
    TerrainVoxel("none", 0, 0x00000000),
    TerrainVoxel("basalt", 8, 0x332c2bff),
    TerrainVoxel("bedrock", 6, 0x4e4a54ff),
    TerrainVoxel("birch_log", 12, 0xd4cdc5ff),
    TerrainVoxel("clay", 17, 0x6e554fff),
    TerrainVoxel("coal_ore", 19, 0x282e2aff),
    TerrainVoxel("cobblestone", 5, 0x524535ff),
    TerrainVoxel("diamond_ore", 23, 0x89c5d9ff),
    TerrainVoxel("dirt", 2, 0x543919ff),
    TerrainVoxel("gold_ore", 11, 0xdecb52ff),
    TerrainVoxel("granite", 13, 0x636d70ff),
    TerrainVoxel("grass", 1, 0x319436ff),
    TerrainVoxel("gravel", 36, 0xab9865ff),
    TerrainVoxel("hay", 35, 0xa39f33ff),
    TerrainVoxel("limestone", 9, 0xc3c4b1ff),
    TerrainVoxel("moss", 40, 0x30784eff),
    TerrainVoxel("neptunium_ore", 30, 0x1d4030ff),
    TerrainVoxel("oak_log", 3, 0x4d442cff),
    TerrainVoxel("pumpkin", 10, 0xc78e08ff),
    TerrainVoxel("quartzite", 7, 0xa89c8aff),
    TerrainVoxel("rubber_log", 14, 0x94570cff),
    TerrainVoxel("silver_ore", 25, 0xd8e3e3ff),
    TerrainVoxel("stone", 4, 0xc1c9c8ff),
    TerrainVoxel("snow", 37, 0xf5f5f5ff),
    
    # Flora
    TerrainVoxel("oak_leaf", to_flora_id(1), 0x234a2dff),
    TerrainVoxel("birch_leaf", to_flora_id(2), 0x586b23ff),
    TerrainVoxel("rubber_leaf", to_flora_id(3), 0x176e47ff),
    TerrainVoxel("switch_grass", to_flora_id(4), 0x1ad94aff),
    TerrainVoxel("azalea_flower", to_flora_id(5), 0xb84f9dff),
    TerrainVoxel("bell_flower", to_flora_id(6), 0x5474a8ff),
    TerrainVoxel("dandelion_flower", to_flora_id(7), 0xadb020ff),
    TerrainVoxel("daylily_flower", to_flora_id(8), 0xbf5a17ff),
    TerrainVoxel("lilac_flower", to_flora_id(9), 0x742fb5ff),
    TerrainVoxel("rose_flower", to_flora_id(10), 0xb51200ff),
    TerrainVoxel("cotton_flower", to_flora_id(11), 0xd1c9c7ff),
    TerrainVoxel("hemp_bush", to_flora_id(12), 0x18381eff),
    TerrainVoxel("red_mushroom", to_flora_id(13), 0x82283fff),
]

terrain_index = {tv.name: tv.index for tv in terrain}

terrain_color = np.zeros(max(tv.index for tv in terrain) + 1, dtype=np.uint32)
for tv in terrain:
    terrain_color[tv.index] = tv.color

In [ ]:
@dataclass
class UndegroundMap:
    ids: Dict[str, int]
    colors: List[int]
    assignments: np.ndarray

In [ ]:
@dataclass
class RegionMap:
    ids: Dict[str, int]
    assignments: np.ndarray
    colors: List[int]
    radii: List[int]
        
@dataclass
class HeightMap:
    data: np.ndarray
        
@dataclass
class Column:
    name: str
    samples: List[np.ndarray]
    color: int

@dataclass
class SurfaceMap:
    ids: Dict[str, int]
    assignments: np.ndarray
    columns: List[Column]
        
@dataclass
class ColumnMap:
    data: np.ndarray

In [ ]:
@dataclass
class Surface:
    rm: Any
    hm: Any
    sm: Any
    cm: Any

In [ ]:
@dataclass
class TreeMap:
    trees: List[np.array]
    assignments: np.array
        
@dataclass
class FloraMap:
    assignments: np.ndarray

In [ ]:
def load_surface(chunk):
    name = "_".join(map(str, chunk))
    with gzip.open(f"F:/Biomes/alpha_world/surface/regions_{name}.dump", "rb") as f:
        rm = pickle.load(f)
    with gzip.open(f"F:/Biomes/alpha_world/surface/heights_{name}.dump", "rb") as f:
        hm = pickle.load(f)
    with gzip.open(f"F:/Biomes/alpha_world/surface/surface_{name}.dump", "rb") as f:
        sm = pickle.load(f)
    with gzip.open(f"F:/Biomes/alpha_world/surface/columns_{name}.dump", "rb") as f:
        cm = pickle.load(f)
    return Surface(rm, hm, sm, cm)

In [ ]:
surface = load_surface((0, 0))

In [ ]:
def load_underground(chunk):
    name = "_".join(map(str, chunk))
    with gzip.open(f"F:/Biomes/alpha_world/underground/um_{name}.dump", "rb") as f:
        return pickle.load(f)

In [ ]:
def get_cave_filter(chunk, shape):
    z_dim, _, x_dim = shape
    nm = explicit_noise(
        (z_dim, 16, x_dim),
        64,
        [1, 1, 1],
        transform=lambda coords : coords + [chunk[0] * z_dim, 0, chunk[1] * x_dim],
        seed="cave_filter"
    )
    return nm < 0.5

In [ ]:
def to_voxels(um):
    index = np.array([terrain_index[name] for name, i in um.ids.items()], dtype=int)
    return index[um.assignments]

In [ ]:
with gzip.open("F:/Biomes/alpha_world/features/trees.dump", "rb") as f:
    tm = pickle.load(f)
    
with gzip.open("F:/Biomes/alpha_world/features/florae.dump", "rb") as f:
    fm = pickle.load(f)

In [ ]:
## TODO: This code does the wrong think at chunk boundaries. Modify it
# to include trees from neighboring areas.

def add_trees(chunk, final_map):   
    tm_z0 = chunk[0] * final_map.shape[0]
    tm_x0 = chunk[1] * final_map.shape[2]
    tm_z1 = tm_z0 + final_map.shape[0]
    tm_x1 = tm_x0 + final_map.shape[2]
    
    assignments = tm.assignments[tm_z0:tm_z1, tm_x0:tm_x1]
    
    hm = (final_map != 0).argmax(axis=1)
    for z, x in np.stack(np.where(assignments), axis=-1):    
        if z >= hm.shape[0] or x >= hm.shape[1]:
            continue
            
        h = hm[z, x]
        v = final_map[z, h, x] 
        if v not in [terrain_index["moss"], terrain_index["grass"]]:
            continue
        
        tree = tm.trees[assignments[z, x] - 1][:, ::-1, :]
        mask = tree != 0
        
        sz, sy, sx = tree.shape
        z0, z1 = z - sz // 2, z + sz // 2
        y0, y1 = h - sy, h
        x0, x1 = x - sx // 2, x + sx // 2
        
        if x0 < 0 or y0 < 0 or z0 < 0:
            continue
        
        if z1 >= final_map.shape[0] or y1 >= final_map.shape[1] or x1 >= final_map.shape[2]:
            continue

        if np.all(final_map[z0:z1, y0:y1, x0:x1][mask] == 0):
            final_map[z0:z1, y0:y1, x0:x1][mask] = tree[mask]

In [ ]:
def add_flora(chunk, final_map):
    z0 = chunk[0] * final_map.shape[0]
    x0 = chunk[1] * final_map.shape[2]
    z1 = z0 + final_map.shape[0]
    x1 = x0 + final_map.shape[2]
    
    assignments = fm.assignments[z0:z1, np.newaxis, x0:x1]
    
    # Generate grass mask.
    shape = (final_map.shape[0], final_map.shape[2])
    mask = assignments != 0
    
    # Extract the top surface voxels.
    index = (final_map != 0).argmax(axis=1, keepdims=True)
    values = np.take_along_axis(final_map, index, axis=1)
    
    # Update mask to restrict to certain surface voxels.
    mask = np.logical_and(
        mask,
        np.logical_or(
            values == terrain_index["grass"],
            values == terrain_index["moss"],
        )
    )
    
    # Work out the values to write.
    values[np.logical_not(mask)] = 0
    values[mask] = assignments[mask]
    
    # Write the values into the map.
    index = np.clip(index - 1, 0, final_map.shape[1])
    np.put_along_axis(final_map, index, values, axis=1)

In [ ]:
def save_final_map(chunk, final_map):
    nz = final_map.shape[0] // 32
    ny = final_map.shape[1] // 32
    nx = final_map.shape[2] // 32
    
    # TODO: Speed this up. The inner three loops should run in C++ directly over the dense array.
    for bz in range(nz):
        for by in range(ny):
            for bx in range(nx):
                block = voxeloo.biomes.VolumeBlock_U32()
                for z in range(32):
                    sz = 32 * bz + z
                    for y in range(32):
                        sy = 32 * by + y
                        for x in range(32):
                            sx = 32 * bx + x
                            block[x, 31 - y, z] = final_map[sz, sy, sx]
                blob = block.dumps()
                name = f"{chunk[0] * nz + bz}_{by}_{chunk[1] * nx + bx}"
                path = f"F:/Biomes/alpha_world/shards/block_{name}"
                with open(path, "w") as f:
                    f.write(blob)

In [ ]:
chunks = [(i, j) for i in range(16) for j in range(16)]

In [ ]:
%%time

projected_maps = []

for chunk in tqdm(chunks):
    final_map = to_voxels(load_underground((chunk[0], 0, chunk[1]))) 
    cave_filter = get_cave_filter(chunk, final_map.shape)

    # Chop off the top by applying the surface columns:
    for i in range(final_map.shape[0]):
        for j in range(final_map.shape[2]):
            z = chunk[0] * final_map.shape[0] + i
            x = chunk[1] * final_map.shape[2] + j

            h = 256 - int(surface.hm.data[z, x])
            c = surface.cm.data[z, :, x]

            m = np.logical_and(
                c != 0,
                np.logical_or(
                    final_map[i, h:h + 16, j] != 0,
                    cave_filter[i, :, j],
                )
            )

            final_map[i, :h, j] = 0
            final_map[i, h:h + 16, j][m] = c[m]
            
    
    # Add flora.
    add_flora(chunk, final_map)
            
    # Add trees.
    add_trees(chunk, final_map)
    
    # Save the chunk.
    save_final_map(chunk, final_map)
    
    # Append the projected top of the chunk for the minimap.
    projected_maps.append(
        np.take_along_axis(
            final_map, 
            (final_map != 0).argmax(axis=1, keepdims=True),
            axis=1,
        )
    )

In [ ]:
visualize_voxels(terrain_color[final_map])

In [ ]:
z_s = (1 + chunks[-1][0]) * projected_maps[-1].shape[0]
x_s = (1 + chunks[-1][1]) * projected_maps[-1].shape[2]

minimap = np.zeros(shape=(z_s, 1, x_s), dtype=int)
for pm, chunk in tqdm(zip(projected_maps, chunks)):
    z0 = chunk[0] * pm.shape[0]
    x0 = chunk[1] * pm.shape[2]
    z1 = z0 + pm.shape[0]
    x1 = x0 + pm.shape[2]
    
    minimap[z0:z1, :, x0:x1] = pm

In [ ]:
visualize_voxels(terrain_color[minimap])